# SA-1B Reader, Visualization And Puzzles Demo

这个 notebook 调用 /src/sa1b 下的读取、可视化与 puzzles 桥接模块，完成单样本读取、标注关联、掩码叠加、bbox 展示、多样本抽样查看，以及基于前景 mask 的子图切分演示。

In [ ]:
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

project_root = Path.cwd()
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from sa1b.dataset import SA1BReader
from sa1b.puzzles_bridge import generate_puzzle_crops
from sa1b.visualization import plot_sample, plot_sample_grid

DATA_ROOT = Path('/mnt/dataset/marko/data/sa1b/data')
SHARD_NAME = 'sa_000000'
reader = SA1BReader(DATA_ROOT)
plt.rcParams['figure.figsize'] = (12, 8)

In [ ]:
sample_stems = reader.list_sample_stems(SHARD_NAME)
print(f'shard: {SHARD_NAME}')
print(f'sample count: {len(sample_stems)}')
print('first 10 samples:', sample_stems[:10])

In [ ]:
sample_stem = 'sa_16'
sample = reader.load_sample(SHARD_NAME, sample_stem)
image = reader.load_image(SHARD_NAME, sample_stem)

print('image path:', sample.image_path)
print('annotation path:', sample.annotation_path)
print('image size:', image.size)
print('annotation count:', len(sample.annotations))
print('first bbox:', sample.annotations[0].bbox if sample.annotations else None)
print('first stability_score:', sample.annotations[0].stability_score if sample.annotations else None)

In [ ]:
ax = plot_sample(
    sample,
    image=image,
    max_annotations=200,
    show_bboxes=True,
    show_points=True,
    alpha=0.35,
    seed=16,
)
plt.show()

In [ ]:
random.seed(42)
picked = random.sample(sample_stems, 4)
print('picked samples:', picked)
plot_sample_grid(
    reader,
    SHARD_NAME,
    picked,
    max_annotations=50,
    columns=2,
    figsize=(16, 12),
    seed=42,
)
plt.show()

## Puzzles 子图切分演示

下面基于 SA-1B 的分割标注先合成前景 mask，再调用 puzzles 的切框逻辑，把一张完整图片切分成多个受 mask 约束的子图。

In [ ]:
puzzle_result = generate_puzzle_crops(
    sample,
    image=image,
    n_boxes=6,
    min_box_size_scale=0.2,
    max_box_size_scale=0.45,
    min_overlap=0.1,
    max_overlap=0.3,
    min_mask_coverage=0.25,
    max_annotations=120,
    min_area=1024.0,
    min_predicted_iou=0.85,
    min_stability_score=0.9,
    seed=16,
 )

print('sample:', puzzle_result.sample_stem)
print('kept annotation count:', len(puzzle_result.kept_annotation_ids))
print('foreground pixels:', int(puzzle_result.foreground_mask.sum()))
print('template box:', puzzle_result.template_box)
print('generated boxes:', puzzle_result.boxes)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
axes[0].imshow(puzzle_result.foreground_mask, cmap='gray')
axes[0].set_title('union foreground mask')
axes[0].axis('off')

axes[1].imshow(puzzle_result.image)
for index, (x1, y1, x2, y2) in enumerate(puzzle_result.boxes):
    color = plt.cm.tab10(index % 10)
    axes[1].add_patch(
        Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=2.0, edgecolor=color)
    )
    axes[1].text(x1, y1, f'#{index}', color='white', fontsize=10, bbox={'facecolor': color, 'alpha': 0.75, 'pad': 2})
axes[1].set_title('puzzles boxes on SA-1B image')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image
from matplotlib.patches import Patch

from sa1b.visualization import decode_segmentation

columns = 2
samples_per_row = 2
rows = int(np.ceil(len(puzzle_result.resized_crops) / samples_per_row))
fig, axes = plt.subplots(rows, samples_per_row * columns, figsize=(14, 4.5 * rows))
axes = np.atleast_2d(axes)

template_height = puzzle_result.template_box[3] - puzzle_result.template_box[1]
template_width = puzzle_result.template_box[2] - puzzle_result.template_box[0]

kept_id_set = set(puzzle_result.kept_annotation_ids)
kept_annotations = [
    annotation for annotation in sample.annotations if annotation.annotation_id in kept_id_set
]
annotation_ids = [annotation.annotation_id for annotation in kept_annotations]
annotation_id_to_color_index = {
    annotation_id: color_index for color_index, annotation_id in enumerate(annotation_ids, start=1)
}

mask_labels = np.zeros(puzzle_result.foreground_mask.shape, dtype=np.uint16)
for annotation in kept_annotations:
    annotation_mask = decode_segmentation(annotation.segmentation, as_bool=True)
    color_index = annotation_id_to_color_index[annotation.annotation_id]
    mask_labels[annotation_mask] = color_index

mask_colors = plt.cm.tab20(np.linspace(0, 1, max(len(annotation_ids), 1)))
box_annotation_ids = []
for x1, y1, x2, y2 in puzzle_result.boxes:
    crop_labels = mask_labels[y1:y2, x1:x2]
    present_ids = [
        annotation_id
        for annotation_id, color_index in annotation_id_to_color_index.items()
        if np.any(crop_labels == color_index)
    ]
    box_annotation_ids.append(present_ids)

displayed_ids = sorted({annotation_id for present_ids in box_annotation_ids for annotation_id in present_ids})
legend_handles = [
    Patch(
        facecolor=mask_colors[annotation_id_to_color_index[annotation_id] - 1],
        edgecolor='none',
        label=str(annotation_id),
    )
    for annotation_id in displayed_ids
]

for index, crop in enumerate(puzzle_result.resized_crops):
    row, sample_column = divmod(index, samples_per_row)
    image_axis = axes[row, sample_column * columns]
    mask_axis = axes[row, sample_column * columns + 1]
    x1, y1, x2, y2 = puzzle_result.boxes[index]
    crop_labels = mask_labels[y1:y2, x1:x2]
    resized_labels = np.array(
        Image.fromarray(crop_labels).resize(
            (template_width, template_height),
            resample=Image.Resampling.NEAREST,
        )
    )

    colored_mask = np.zeros((template_height, template_width, 4), dtype=np.float32)
    for color_index, color in enumerate(mask_colors, start=1):
        colored_mask[resized_labels == color_index] = (*color[:3], 1.0)

    present_ids = box_annotation_ids[index]

    image_axis.imshow(crop)
    image_axis.set_title(f'#{index} image')
    image_axis.axis('off')

    mask_axis.imshow(colored_mask)
    mask_axis.set_title(f'#{index} mask ({len(present_ids)} ids)')
    mask_axis.axis('off')

for index in range(len(puzzle_result.resized_crops), rows * samples_per_row):
    row, sample_column = divmod(index, samples_per_row)
    axes[row, sample_column * columns].axis('off')
    axes[row, sample_column * columns + 1].axis('off')

fig.suptitle('puzzle crops and color-coded annotation masks', fontsize=16)
fig.tight_layout()
plt.show()

if legend_handles:
    legend_columns = min(4, len(legend_handles))
    legend_rows = int(np.ceil(len(legend_handles) / legend_columns))
    legend_fig_height = max(1.5, 0.45 * legend_rows + 0.6)
    legend_fig, legend_ax = plt.subplots(figsize=(14, legend_fig_height))
    legend_ax.axis('off')
    legend_fig.legend(
        handles=legend_handles,
        title='annotation id to color',
        loc='center',
        ncol=legend_columns,
        frameon=True,
    )
    legend_fig.tight_layout()
    plt.show()